In [ ]:
!pip install -U ddgs
%pip install wikipedia
%pip install sqlalchemy
%pip install numexpr

In [1]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_ollama import OllamaEmbeddings, ChatOllama

from langchain_ibm import WatsonxEmbeddings
from langchain_ibm import ChatWatsonx

from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain.agents import create_agent

from langchain_community.tools import DuckDuckGoSearchRun


/Users/sunahgwak/Documents/Repository/source/ollama/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

apiKey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.environ["HF_TOKEN"]
COHERE_API_KEY = os.environ["COHERE_API_KEY"]
SERPER_API_KEY = os.getenv("SERPER_API_KEY")

watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apiKey}",
    project_id=f"{project_id}"
)

watson_llm = ChatWatsonx(
  model_id="ibm/granite-4-h-small",
  url=f"{watsonx_ai_url}",
  api_key = f"{apiKey}",
  project_id=f"{project_id}",
  max_tokens = 2000,
  params = {
    "temperature":0
  }
)

ollama_embedding = OllamaEmbeddings(model="nomic-embed-text-v2-moe")

hugging_llm = ChatOpenAI(
  base_url = "https://router.huggingface.co/v1",
  api_key=hf_token,
  model="Qwen/Qwen2.5-7B-Instruct:together",
  temperature=0
)

qwen_llm = ChatOllama(model="qwen3.5:4b", temperature=0)


### Agent
#### 1. 내장 Tool 사용

In [ ]:
# Search Tool - DuckDuckGoSearch(무료, api 키 발급 X)

response = watson_llm.invoke("2026 대한민국 대통령은 누구인가요?")
print(response.content)

'2026년 대한민국 대통령에 대한 정보는 현재로서는 제공할 수 없습니다. 대통령 선거는 일반적으로 5년마다 치러지며, 다음 선거는 2027년에 예정되어 있습니다. 따라서 2026년에 대한민국 대통령이 누구가 될지는 아직 결정되지 않았습니다. 선거 결과는 선거가 진행되고 나서야 알 수 있습니다.'

In [12]:
response = watson_llm.invoke("2024년 노벨 물리학상 수장자의 나이는 몇 살인가요?")
print(response.content)

죄송합니다. 현재 2024년 노벨 물리학상 수상자는 아직 발표되지 않았습니다. 노벨상 수상자는 매년 10월에 발표되며, 수상자의 나이는 그때 발표되는 정보에 따라 알 수 있습니다.


In [22]:
from langchain_community.tools import DuckDuckGoSearchRun

tools = DuckDuckGoSearchRun()

agent = create_agent(
  model = watson_llm,
  tools = [tools],
  system_prompt="You are a helpful assistant"
)

result = agent.invoke(
  
  {
    "messages": [{
      "role":"user",
      "content": "2026 대한민국 대통령은 누구인가요?"
    }]
  }
)

result


{'messages': [HumanMessage(content='2026 대한민국 대통령은 누구인가요?', additional_kwargs={}, response_metadata={}, id='65576722-498b-4b06-85fa-53e720cf74ef'),
  AIMessage(content='2026년 대한민국 대통령 선거는 아직 열리지 않았기 때문에 현재 누구가 대통령이 될지를 확정할 수 없습니다.  \n\n대통령 선거는 5년마다 치러지며, 현재 문재인 대통령의 임기는 2022년 5월 10일에 종료되었습니다. 그 다음 대통령 선거는 **2027년 3월**에 예정돼 있습니다.  \n\n따라서 2026년에는 아직 대통령이 선출되지 않았고, 2027년 3월에 실시되는 선거를 통해 새로운 대통령이 선출될 예정입니다. 현재까지 공식적으로 발표된 후보자나 당선자는 없습니다.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 246, 'prompt_tokens': 241, 'total_tokens': 487}, 'model_name': 'ibm/granite-4-h-small', 'system_fingerprint': '', 'finish_reason': 'stop'}, id='chatcmpl-818fe5b22950ccdd7659b9e7e823cfb2', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 241, 'output_tokens': 246, 'total_tokens': 487})]}

In [32]:
def print_result(result):
  for i, msg in enumerate(result["message"]):
    print("="*50)
    print(msg.get("HumanMessage")["content"])
    print("="*50)


- 구글 웹서치
  - https://serper.dev/ 회원가입 키 발급

In [ ]:
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import Tool

search = GoogleSerperAPIWrapper()

search_tool = Tool(
  name="google_search",
  func=search.run,
  description="""google 검색 최신. 뉴스, 트렌드, 최신 기술 동향 검색에 사용하세요."""
)

agent = create_agent(
  model = hugging_llm,
  tools = [search_tool],
  system_prompt="You are a helpful assistant"
)

result = agent.invoke(
  
  {
    "messages": [{
      "role":"user",
      "content": "2026 대한민국 대통령은 누구인가요?"
    }]
  }
)

result

- Wikipedia

In [ ]:
%pip install wikipedia

In [4]:
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

tools = WikipediaQueryRun(api_wrapper = WikipediaAPIWrapper())

agent = create_agent(
  model = qwen_llm,
  tools = [tools],
  system_prompt="You are a helpful assistant"
)

result = agent.invoke(
  {
    "messages": [{
      "role":"user",
      "content": "앨런 튜링은 누구인가요?"
    }]
  }
)

result

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [3]:
def print_result(result):
    for i, msg in enumerate(result['messages']):
        print(f"\n===== {i} ====")
        print(type(msg).__name__)

        if hasattr(msg, "content"):
            print(msg.content)

        if hasattr(msg, "tool_calls"):
            print(msg.tool_calls)

In [11]:
from langchain_community.tools import ListDirectoryTool, ReadFileTool, WriteFileTool, CopyFileTool, DeleteFileTool

tools = [ReadFileTool(), WriteFileTool(), ListDirectoryTool(), DeleteFileTool()]

agent = create_agent(
  model=watson_llm,
  tools =tools,
)

result = agent.invoke({
  "messages": [{
      "role":"user",
      # "content": "현재 파일 목록 보여줘",
      # "content": "news.py 파일의 내용을 보여줘",
      # "content":"현재 파일 목록을 summary.txt파일로 저장시켜줘",
      # "content":"현재 파일 목록을 summary2.txt로 저장하고 files.txt로 복사한 후 저장해줘",
      'content': "현재 폴더의 files.txt파일 삭제해줘"
    }]
})

print(result)

{'messages': [HumanMessage(content='현재 폴더의 files.txt파일 삭제해줘', additional_kwargs={}, response_metadata={}, id='9481fade-d1a7-4937-9ce7-044b0015a773'), AIMessage(content='', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 448, 'total_tokens': 477}, 'model_name': 'ibm/granite-4-h-small', 'system_fingerprint': '', 'finish_reason': 'tool_calls'}, id='chatcmpl-753b9df1aa3163b75203ab5606854e87', tool_calls=[{'name': 'file_delete', 'args': {'file_path': 'files.txt'}, 'id': 'chatcmpl-tool-89c65c97cf2c13fb', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 448, 'output_tokens': 29, 'total_tokens': 477}), ToolMessage(content='File deleted successfully: files.txt.', name='file_delete', id='bba374a1-df6c-414f-907a-14bf2e3b0c06', tool_call_id='chatcmpl-tool-89c65c97cf2c13fb'), AIMessage(content='현재 폴더에 있는 `files.txt` 파일이 성공적으로 삭제되었습니다.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'p

- 데이터베이스 agent

In [8]:
# db생성
import sqlite3

conn = sqlite3.connect("./db/company.db")
cursor = conn.cursor()
cursor.execute("drop table if exists employee")

cursor.execute("""
CREATE TABLE employee(id integer primary key autoincrement, name text, salary integer)
""")

cursor.execute("""
INSERT INTO employee(name, salary)
VALUES
('KIM',4000),
('LEE',5000),
('Park',6000)
""")

conn.commit()
conn.close()

In [6]:
%pip install sqlalchemy


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
# sqlalchemy: python에서 데이터베이스를 편리하게 다루기 위한 라이브러리
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///db/company.db")
print(db.run("select * from employee"))

[(1, 'KIM', 4000), (2, 'LEE', 5000), (3, 'Park', 6000)]


In [15]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db = db, llm = watson_llm)
tools = toolkit.get_tools()

agent = create_agent(
  model=watson_llm,
  tools=tools,
  system_prompt="You are a database assistant."
)

result = agent.invoke({
  "messages":[
    {
      "role":"user",
      "content":"직원의 평균 급여는 얼마인가요?"
    }
  ]
})

print_result(result)


===== 0 ====
HumanMessage
직원의 평균 급여는 얼마인가요?

===== 1 ====
AIMessage

[{'name': 'sql_db_list_tables', 'args': {'tool_input': ''}, 'id': 'chatcmpl-tool-ae3a2bc41610e254', 'type': 'tool_call'}]

===== 2 ====
ToolMessage
employee

===== 3 ====
AIMessage

[{'name': 'sql_db_schema', 'args': {'table_names': 'employee'}, 'id': 'chatcmpl-tool-b84ae42b2cd4064e', 'type': 'tool_call'}]

===== 4 ====
ToolMessage

CREATE TABLE employee (
	id INTEGER, 
	name TEXT, 
	salary INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from employee table:
id	name	salary
1	KIM	4000
2	LEE	5000
3	Park	6000
*/

===== 5 ====
AIMessage

[{'name': 'sql_db_query', 'args': {'query': 'SELECT AVG(salary) AS avg_salary FROM employee;'}, 'id': 'chatcmpl-tool-90af3a9a5f2893b4', 'type': 'tool_call'}]

===== 6 ====
ToolMessage
[(5000.0,)]

===== 7 ====
AIMessage
직원들의 평균 급여는 **5,000**입니다.
[]


- calculator
  - 계산

In [17]:
%pip install numexpr


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [20]:
from langchain_classic.chains import LLMMathChain
from langchain_core.tools  import Tool

llm_math = LLMMathChain.from_llm(llm=watson_llm, verbose=True)
calc_tool = Tool(
  name="calculator", 
  func=llm_math.run, 
  description="수학 계산이 필요할 때 사용합니다. 시장 규모 계산, 성장률 계산, 단위 변화 등에 사용하세요."
)

calc_tool.invoke("1200억 달러를 원화로 환산해줘(1달러 1530원)")



> Entering new LLMMathChain chain...
1200억 달러를 원화로 환산해줘(1달러 1530원)Question: 1200억 달러를 원화로 환산해줘(1달러 1530원)
```text
120000000000 * 1530
```
...numexpr.evaluate("120000000000 * 1530")...
```output
183600000000000
```
Answer: 1200억 달러는 1달러당 1530원 환율로 환산하면 183,600,000,000,000원입니다.
> Finished chain.


'Answer:  1200억 달러는 1달러당 1530원 환율로 환산하면 183,600,000,000,000원입니다.'

#### 커스텀 Tool
- description 작성
  - 기능 설명, 언제 사용해야 하는지, 입력값 형식, 출력값 형식, 필수 or 선택 명시
- 작성 방법
  - @tool 데코레이터
  - StructedTool
  - BaseTool

In [22]:
from langchain_core.tools import tool

# 함수 구현
@tool
def calculator(expression):
  """
  수식을 계산합니다.
  사칙연산에 사용하세요.
  """
  return str(eval(expression))

print(calculator.invoke("123 * 567"))
print(calculator.name)
print(calculator.description)

69741
calculator
수식을 계산합니다.
사칙연산에 사용하세요.


In [24]:
# 사칙연산 합수를 agent에 붙이고 싶다
agent = create_agent(
  model=watson_llm, 
  tools=[calculator]
)
result = agent.invoke({
  "messages":[
    {
      "role":"user",
      "content":"123 곱하기 567는?"
    }
  ]
})
print_result(result)


===== 0 ====
HumanMessage
123 곱하기 567는?

===== 1 ====
AIMessage

[{'name': 'calculator', 'args': {'expression': '123 * 567'}, 'id': 'chatcmpl-tool-90cddad6d4bd75f5', 'type': 'tool_call'}]

===== 2 ====
ToolMessage
69741

===== 3 ====
AIMessage
123 곱하기 567의 결과는 **69,741**입니다.
[]


In [28]:
from pydantic import BaseModel, Field

class ExchangeRateInput(BaseModel):
  from_currency:str = Field(description='변환할 통화 코드 (예: USD, EUR, JPY, KRW)')
  to_currency:str = Field(description='변환된 통화 코드 (예: KRW, USD)')
  amount: float = Field(default=1.0, description="변화할 금액 (기본값: 1)")

# tool로 들어오는 함수는 
# Function must have a docstring if description not provided.
@tool(args_schema=ExchangeRateInput)
def get_exchange_rate(from_currency, to_currency, amount):
  """
  실시간 환율을 조회하고 금액을 변환합니다.
  USD, EUR, JPY, KRW 통화간 환율 계산에 사용하세요
  해외결제, 환전, 가격 비교 질문에 사용합니다.
  """
  mock_rates = {
    ('USD', 'KRW'):1540.0,
    ('EUR', 'KRW'):1440.0,
    ('JPY', 'KRW'):9.2,
    ('KRW', 'USD'):0.00064
  }
  
  rate = mock_rates.get((from_currency, to_currency))
  if not rate:
    return f"{from_currency} -> {to_currency} 환율 정보 없음"
  converted = amount * rate
  return f"{amount} {from_currency} = {converted: .2f} {to_currency} (환율: {rate})"

In [ ]:
agent = create_agent(
  model=watson_llm, 
  tools=[get_exchange_rate]
)
result = agent.invoke({
  "messages":[
    {
      "role":"user",
      "content":"1달라 한국돈으로 환전하면 얼마인가요?"
    }
  ]
})
print_result(result)


===== 0 ====
HumanMessage
1달라 한국돈으로 환전하면 얼마인가요?

===== 1 ====
AIMessage

[{'name': 'get_exchange_rate', 'args': {'amount': 1, 'from_currency': 'USD', 'to_currency': 'KRW'}, 'id': 'chatcmpl-tool-9f403266628f15e3', 'type': 'tool_call'}]

===== 2 ====
ToolMessage
1.0 USD =  1540.00 KRW (환율: 1540.0)

===== 3 ====
AIMessage
1달러(USD)를 한국 원(KRW)으로 환전하면 **약 1,540원**입니다. (현재 환율: 1 USD ≈ 1,540 KRW)
[]


In [4]:
from langchain_core.tools import StructuredTool

def multiply(a:int, b:int) -> int:
  """두 숫자를 곱합니다."""
  return a*b

# 이미 존재하는 파이썬 함수를 Tool로 변환
multi_tool = StructuredTool.from_function(func=multiply)

agent = create_agent(
  model=watson_llm, 
  tools=[multi_tool]
)
result = agent.invoke({
  "messages":[
    {
      "role":"user",
      "content":"12 곱하기 34는?"
    }
  ]
})
print_result(result)



===== 0 ====
HumanMessage
12 곱하기 34는?

===== 1 ====
AIMessage

[{'name': 'multiply', 'args': {'a': 12, 'b': 34}, 'id': 'chatcmpl-tool-a8eb7a3c293611a6', 'type': 'tool_call'}]

===== 2 ====
ToolMessage
408

===== 3 ====
AIMessage
12를 34로 곱하면 **408**입니다.
[]


In [6]:
def calculate_bmi(height:float, weight:float) -> str:
  """ 
  height(cm)과 weight(kg)를 입력받아 BMI 계산
  """
  bmi = weight / ((height / 100) ** 2)
  return f"BMI = {bmi:.2f}"
  

In [7]:
bmi_tool = StructuredTool.from_function(calculate_bmi)

agent = create_agent(
  model=watson_llm, 
  tools=[bmi_tool]
)
result = agent.invoke({
  "messages":[
    {
      "role":"user",
      "content":"키는 180에 몸무게 100키로"
    }
  ]
})
print_result(result)



===== 0 ====
HumanMessage
키는 180에 몸무게 100키로

===== 1 ====
AIMessage

[{'name': 'calculate_bmi', 'args': {'height': 180, 'weight': 100}, 'id': 'chatcmpl-tool-a396eeaf1cd494c3', 'type': 'tool_call'}]

===== 2 ====
ToolMessage
BMI = 30.86

===== 3 ====
AIMessage
당신의 BMI는 30.86입니다. 이는 과체중(Obesity) 범주에 해당합니다. 건강한 체중을 유지하기 위해 균형 잡힌 식단과 꾸준한 운동을 권장합니다.
[]


In [ ]:
from langchain_core.tools import BaseTool

class WeatherInput(BaseModel):
  city:str = Field(description="날씨를 조회할 도시명")

class WeatherTool(BaseTool):
  name:str = "weather"
  description:str = "특정 도시의 날짜를 조회합니다."
  
  args_schema:type[BaseModel] = WeatherInput
  
  def _run(self, city:str):
    weather_data = {
      "서울": "맑음, 30도",
      "부산": "맑음, 32도",
      "제주": "비, 20도",
      "강원": "흐림, 25도",
    }
    
    return weather_data.get(city, f"{city} 날씨 정보 없음")
  
  
  

In [ ]:

agent = create_agent(
  model=watson_llm, 
  tools=[bmi_tool]
)
result = agent.invoke({
  "messages":[
    {
      "role":"user",
      "content":"서울 날씨는?"
    }
  ]
})
print_result(result)

In [4]:
# Retriever Tool

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_core.tools import create_retriever_tool

docs = [
  Document(page_content="연차는 입사 1년 미만은 월 1개 발생합니다."),
  Document(page_content="병가는 연 30일까지 사용 가능합니다."),
  Document(page_content="퇴직금은 1년 이상 근무 시 지급됩니다."),
]

vectorstore = Chroma.from_documents(docs, embedding=watson_embedding)
retriever = vectorstore.as_retriever()

# retriever를 툴로 등록
retriever_tool = create_retriever_tool(retriever, name="company_policy", description="회사 인사 규정을 검색합니다.")


In [5]:
agent = create_agent(
  model=watson_llm, 
  tools=[retriever_tool]
)
result = agent.invoke({
  "messages":[
    {
      "role":"user",
      "content":"연차는 어떻게 발생하나요?"
    }
  ]
})
# print_result(result)

In [3]:
from langchain.agents import create_agent
load_dotenv()


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


agent = create_agent(
    model=watson_llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in San Francisco?"}]}
)

{'messages': [HumanMessage(content='What is the weather in San Francisco?', additional_kwargs={}, response_metadata={}, id='66ea6696-4236-4e77-a284-ecf9ddb089a9'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 194, 'total_tokens': 222}, 'model_name': 'ibm/granite-4-h-small', 'system_fingerprint': '', 'finish_reason': 'tool_calls'}, id='chatcmpl-ac504a6b82d5bf44ed4fa0cf3902c467', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'chatcmpl-tool-a538553951f271cc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 194, 'output_tokens': 28, 'total_tokens': 222}),
  ToolMessage(content="It's always sunny in San Francisco!", name='get_weather', id='5654dada-ef75-4571-a0b0-7259a133ea59', tool_call_id='chatcmpl-tool-a538553951f271cc'),
  AIMessage(content="It's always sunny in San Francisco!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tok

#### 리서치 자동화 Agent 구현
- 입력: 리서치 주제
- tool
  - 1. 리서치 주제에 대해 웹 검색으로 최신 동향 수집
  - 2. 최신 동향 요약
  - 3. 수치 데이터가 있다면 계산기로 검증/변환
  - 4. 마크다운 형식의 문서로 저장

In [7]:
# 1. 웹 검색이 가능한 tool(구글)
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import Tool

search = GoogleSerperAPIWrapper()

search_tool = Tool(
  name="google_search",
  func=search.run,
  description="""google 검색 최신. 뉴스, 트렌드, 최신 기술 동향 검색에 사용하세요."""
)

result = search_tool.run("AI 반도체 시장 2026년 전망")

In [9]:
from langchain_core.tools import tool

# 2. 요약
@tool
def trend_analysis_tool(text):
  """
  검색 결과에서 핵심 트렌드를 분석합니다.
  """
  
  prompt = f"""
  아래 검색 결과를 분석하여
  
  1. 핵심 트렌드
  2. 주요 기업
  3. 성장 요인
  4. 위험 요소
  
  를 정리하세요.
  
  검색결과:
  {text}
  """
  
  return watson_llm.invoke(prompt).content

trend_analysis_tool.invoke(result)

"Based on the provided search results, here is the analysis:\n\n### 1. Key Trends\n- **Increased Investment in AI Technology**: 73% of respondents plan to expand their spending on AI technology by 2026, marking a 6 percentage point increase from the previous year. This indicates a growing recognition of AI's importance across various industries.\n- **Emphasis on Memory Solutions**: As AI demand expands, the significance of memory solutions is becoming more pronounced. This suggests a trend towards enhancing infrastructure to support advanced AI applications.\n\n### 2. Major Companies\nThe search results do not explicitly mention specific companies. However, the context implies that major players in the AI and memory solution sectors are likely to be involved. Companies that are typically at the forefront of AI technology and memory solutions include:\n- **Tech Giants**: Companies like Google, Microsoft, Amazon, and IBM, which are heavily investing in AI research and development.\n- **M

In [16]:
# 3. 계산기
import numexpr

@tool
def calculator_tool(expression):
  """
  수학 계산용 Tool
  
  사용 예:
  999 * 1530
  (250 / 120)**(1/6) - 1
  100000000 / 1250
  
  CAGR 계산
  한윤 계산
  시장 규모 계산
  
  """
  result = numexpr.evaluate(expression)
  
  return str(result.item())

calculator_tool.invoke("2+3* (4-1)")

'11'

In [18]:
from datetime import datetime

@tool
def save_report_tool(content):
  """
  리서치 결과를 Markdown 파일로 저장합니다.
  최종 리포트 작성이 완료되었을 때 사용하세요.
  입력: Markdown 형식의 리포트 전체 내용
  """
  
  timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
  filename = f'research_report_{timestamp}.md'
  with open(filename,'w',encoding='utf-8') as f:
    f.write(content)
    
  return f"리포트 저장 완료: {filename}"

content="""
리서치 결과를 Markdown으로 저장
"""

save_report_tool.invoke(content)


'리포트 저장 완료: research_report_20260605_131444.md'

In [24]:
def save_report(content):
  """
  리서치 결과를 Markdown 파일로 저장합니다.
  최종 리포트 작성이 완료되었을 때 사용하세요.
  입력: Markdown 형식의 리포트 전체 내용
  """
  
  timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
  filename = f'research_report_{timestamp}.md'
  with open(filename,'w',encoding='utf-8') as f:
    f.write(content)
    
  return filename


In [ ]:
# 툴 사용 여부는 모델이 결정
tools = [search_tool, trend_analysis_tool, calculator_tool, save_report_tool]

system_prompt="""
당신은 전문 리서치 어시스턴트입니다.

작업순서:
1. search_tool 사용
2. trend_analysis_tool 사용
3. 수치는 calculator_tool 사용
4. Markdown 리포트 작성
# 5. 작성한 전체 Markdown 리포트를 save_report_tool에 전달하여 저장
"""


agent = create_agent(
    model=watson_llm,
    tools=tools,
    system_prompt=system_prompt,
)

# Run the agent
result = agent.invoke(
    {"messages": 
      [
        {
          "role": "user", 
          "content": """
          주제: AI 반도체 시장 2026 전망
          
          포함내용:
          - 주요 기업
          - 시장 규모
          - 성장률
          - 핵심 트렌드
          """
        }
      ]
    }
)

# print(result["messages"])

report = result['messages'][-1].content
filename = save_report(report)
print(filename)

research_report_20260605_144959.md
